# Session 4 - Control flow with functions

This notebook is a functional refactor of the *possible solution* in `Session_04.ipynb`. Each step of the solution is now a function with explicit inputs and a return value, so each step can be tested, reused or changed without touching the others. The printed output is identical to the original solution.

Conventions used below:
- **Calculation functions** (`calculate_...`, `filter_...`, `classify_...`, `assign_...`, `accumulate_...`, `check_...`) do no printing. They take data in as arguments and return results.
- **Report functions** (`print_...`) only print. They never change the data.
- **No hidden globals.** Inputs and thresholds are passed as parameters. Rows are copied rather than changed in place.
- **`main()`** runs the steps in order, as the original solution did.


# Session 4 - Control flow examples and notes on file handing

File handling in programming refers to the process of creating, reading, writing, and manipulating files on a computer. It allows programs to store data persistently, retrieve it when needed, and manage file operations efficiently. The key function in python for file handling is the `open()` function, which is used to open a file in different modes such as read, write, append, etc. The Python documentation provides a comprehensive guide on file handling, including examples and best practices at the following url: [Python File Handling Documentation](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files)

`open()` returns a file object, and is most commonly used with two positional arguments and one keyword argument: open(filename, mode, encoding=None). The `filename` is the name of the file you want to open, `mode` specifies the mode in which the file is opened (e.g., 'r' for reading, 'w' for writing, 'a' for appending), and `encoding` specifies the encoding format (e.g., 'utf-8').




### STEP 1: `for-in` loop on file iterator to identify ports and their respective total FOB

This section explains the design of Step 1 in some detail AND it introduces Mermaid diagraming for illustrating the Control Flow implemented by the design.

We want to find out which ports handle the most coffee exports. So, our strategy will be to scan the raw records in our data file 'trase_global_coffee_supply_chain.csv' and to calculate the FOB totals per export port. 

```python
import csv

file_path = 'trase_global_coffee_supply_chain.csv'

port_fob_totals = {} 
# Define an empty dictionary to store the Ports by name and their respective total FOB. We can use a dictionary to store the port names as keys and their corresponding total FOB values as values. This allows us to efficiently accumulate the FOB values for each port as we iterate through the CSV file.

with open(file_path, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    # Iterate through each row in the CSV file. because reader is an iterable, the for loop will iterate over each row until all rows have been processed.

    for row in reader: 
    # this is a pre-test loop, the condition is checked first before executing the loop action. If the condition evaluates to false, the loop exits and the workflow continues to the subsequent process.

        # Extract port and fob from the current row using their respective column labels
        port = row['port_of_export_name']
        fob = row['fob']
        if port and fob: # This checks if both port and fob are both present, if not, skip to the next row
            port_fob_totals[port] = port_fob_totals.get(port, 0.0) + float(fob) 
            
            # The get() method is used to retrieve the current total FOB for the current port. Using port as its key, if the port_fob_totals dictionary entry for [port] has not been encountered before (i.e., it is not found in the dictionary), it initializes it's value to 0.0 and adds the current FOB value.. The float(fob) converts the FOB value from a string to a float for accurate numerical calculations.
            # And so we accumulate the FOB value for the corresponding port in the dictionary. 

print("=== STEP 1: PORT FOB TOTALS ===")
for port_name, total_fob in port_fob_totals.items(): 
# We can infer that port_name and total_fob are variables that represent the key-value pairs in the port_fob_totals dictionary, accessed using the .items() method on port_fob_totals. Because port_fob_totals is iterable, the for loop can iterate over it and return each key-value pair in the dictionary, allowing us to access both the port name and its corresponding total FOB value for printing.

    print(f"Port: {port_name:<30} | Total FOB: ${total_fob:,.2f}") 
    # We can use print's f-string structure to print the name of each port (limiting the string printed to 30 characters) and its corresponding total fob.
```

The Mermaid diagram code is rendered in the notebook if Matt Bierner's Markdown Preview Mermaid Support is installed in your VS Code.

```mermaid
flowchart TD
    Start([Start]) --> OpenFile["Open file and initialize csv.DictReader"]
    OpenFile --> Loop1{"For each row in reader?"}

    Loop1 -- Yes --> Extract["Extract port and fob from row"]
    Extract --> CheckCond{"if port and fob?"}

    CheckCond -- Yes --> Accumulate["port_fob_totals[port] += float(fob)"]
    Accumulate --> Loop1

    CheckCond -- No --> Loop1

    Loop1 -- No (Loop Finished) --> PrintHeader["Print Header: STEP 1: PORT FOB TOTALS"]
    PrintHeader --> Loop2{"For port_name, total_fob in port_fob_totals.items()?"}

    Loop2 -- Yes --> PrintRow["Print formatted Port and Total FOB"]
    PrintRow --> Loop2

    Loop2 -- No (Loop Finished) --> End([End])
```


## Setup

Constants are defined once here and passed into the functions as arguments.

In [1]:
import csv

FILE_PATH = 'trase_global_coffee_supply_chain.csv'
HIGH_VALUE_THRESHOLD = 50000.0   # FOB value (USD) above which a shipment counts as high value
TARGET_QUOTA_TONS = 100.0        # raw-equivalent metric tons the audit sample must reach

## Loading the data

Read the CSV file once into a list of dictionaries. Every step below works from this list.

In [2]:
def load_shipments(file_path):
    """Return every row of the CSV file as a dictionary."""
    with open(file_path, mode='r', encoding='utf-8') as file:
        return list(csv.DictReader(file))

### STEP 1: `for-in` loop on file iterator to map ports & accumulate total FOB

`calculate_port_fob_totals(shipments)` returns a dictionary of total FOB per export port. `print_port_fob_totals(...)` prints it.

In [3]:
def calculate_port_fob_totals(shipments):
    """Return a dictionary mapping each export port to its total FOB value.

    Rows with a missing port or FOB value are skipped.
    """
    port_fob_totals = {}
    for row in shipments:
        port = row['port_of_export_name']
        fob = row['fob']
        if port and fob:
            port_fob_totals[port] = port_fob_totals.get(port, 0.0) + float(fob)
    return port_fob_totals


def print_port_fob_totals(port_fob_totals):
    """Print the total FOB value for each export port."""
    print("=== STEP 1: PORT FOB TOTALS ===")
    for port_name, total_fob in port_fob_totals.items():
        print(f"Port: {port_name:<30} | Total FOB: ${total_fob:,.2f}")

### STEP 2: `if` statement to narrow records > $50,000 FOB valuation

`filter_high_value_shipments(shipments, threshold)` returns the rows whose FOB value is above the threshold. `print_high_value_summary(...)` prints how many rows were kept.

In [4]:
def filter_high_value_shipments(shipments, threshold):
    """Return the shipments whose FOB value is greater than the threshold.

    Rows with an empty FOB value are skipped.
    """
    high_value_shipments = []
    for row in shipments:
        if row['fob'] and float(row['fob']) > threshold:
            high_value_shipments.append(row)
    return high_value_shipments


def print_high_value_summary(high_value_shipments, threshold):
    """Print how many shipments exceed the FOB threshold."""
    print(f"\n=== STEP 2: HIGH-VALUE FILTER ===")
    print(f"Isolated {len(high_value_shipments)} shipments exceeding ${threshold:,.0f} FOB.")

### STEP 3: `match-case` pattern matching on HS6 product codes

`classify_product_types(shipments)` returns copies of the rows with a `product_category` field added. `print_product_classification(...)` prints the classified rows.

In [5]:
def classify_product_types(shipments):
    """Return copies of the shipments with a 'product_category' field based on the HS6 code."""
    classified = []
    for row in shipments:
        match row['hs6']:
            case '090111' | '90111':
                category = "Unroasted Green Coffee Beans"
            case '090112' | '90112':
                category = "Decaffeinated Coffee Beans"
            case '210111' | '210111.0':
                category = "Instant Coffee Extracts & Derivatives"
            case _:
                category = "Other Processed Coffee Derivative"
        classified.append({**row, 'product_category': category})
    return classified


def print_product_classification(classified_shipments):
    """Print the HS6 code, product category, FOB value and destination of each shipment."""
    print("\n=== STEP 3: PRODUCT TYPE CLASSIFICATION ===")
    for row in classified_shipments:
        print(f"HS {row['hs6']} ({row['product_category']}) | FOB: ${float(row['fob']):,.2f} -> Destination: {row['country_of_first_import']}")

### STEP 4: `elif` decision chain for regional compliance protocols

`assign_audit_protocols(shipments)` returns copies of the rows with an `audit_protocol` field added, chosen by economic bloc. `print_compliance_tiers(...)` prints the result.

In [6]:
def assign_audit_protocols(shipments):
    """Return copies of the shipments with an 'audit_protocol' field based on the economic bloc."""
    assigned = []
    for row in shipments:
        bloc = row['country_of_first_import_economic_bloc']
        if bloc == 'EUROPEAN UNION':
            protocol = "EUDR Geolocation & Deforestation Audit"
        elif bloc == 'UNITED STATES':
            protocol = "US CBP Agriculture Inspection"
        elif bloc == 'CHINA (MAINLAND)':
            protocol = "GACC Custom Hygiene Clearance"
        else:
            protocol = "Standard MFN Customs Verification"
        assigned.append({**row, 'audit_protocol': protocol})
    return assigned


def print_compliance_tiers(assigned_shipments):
    """Print the destination, economic bloc and audit protocol of each shipment."""
    print("\n=== STEP 4: REGULATORY COMPLIANCE TIERS ===")
    for row in assigned_shipments:
        print(f"Destination: {row['country_of_first_import']:<15} | Economic Bloc: {row['country_of_first_import_economic_bloc']:<15} | Protocol: {row['audit_protocol']}")

### STEP 5: `while` loop to accumulate audit sample up to 100 raw tons

`accumulate_audit_sample(shipments, target)` takes shipments in order until their raw tonnage reaches the target. It returns the selected batch and the total tonnage. `print_audit_sample_summary(...)` prints the totals.

In [7]:
def accumulate_audit_sample(shipments, target_quota_tons):
    """Take shipments in order until their raw equivalent tonnage reaches the target.

    Returns a tuple of (audit_sample_batch, accumulated_raw_mass).
    """
    audit_sample_batch = []
    accumulated_raw_mass = 0.0
    index = 0
    while accumulated_raw_mass < target_quota_tons and index < len(shipments):
        current_shipment = shipments[index]
        raw_mass = float(current_shipment['mass_tonnes_raw_equivalent'])
        accumulated_raw_mass += raw_mass
        audit_sample_batch.append(current_shipment)
        index += 1
    return audit_sample_batch, accumulated_raw_mass


def print_audit_sample_summary(audit_sample_batch, accumulated_raw_mass):
    """Print how many shipments were selected and their total raw equivalent tonnage."""
    print(f"\n=== STEP 5: AUDIT SAMPLE ACCUMULATION ===")
    print(f"Selected {len(audit_sample_batch)} shipments totaling {accumulated_raw_mass:.2f} raw equivalent metric tons.")

### STEP 6: `if-else` verification of quota completion

`check_quota_met(accumulated_raw_mass, target)` returns `True` when the quota is reached. `print_quota_verification(...)` reports the outcome and, on success, the audit sample breakdown.

In [8]:
def check_quota_met(accumulated_raw_mass, target_quota_tons):
    """Return True if the accumulated raw tonnage meets or exceeds the target."""
    return accumulated_raw_mass >= target_quota_tons


def print_quota_verification(audit_sample_batch, accumulated_raw_mass, target_quota_tons):
    """Print whether the quota was met and, if so, the full audit sample breakdown."""
    print("\n=== STEP 6: QUOTA VERIFICATION ===")
    if check_quota_met(accumulated_raw_mass, target_quota_tons):
        print(f"STATUS: SUCCESS. Target sampling quota of {target_quota_tons} tons met.")
        print("Final Audit Sample Breakdown:")
        for item in audit_sample_batch:
            print(f"  - Exporter: {item['exporter_name'][:25]:<25} | Port: {item['port_of_export_name']:<15} | Raw Tons: {float(item['mass_tonnes_raw_equivalent']):6.2f} t | Protocol: {item['audit_protocol']}")
    else:
        print(f"STATUS: SHORTFALL. Collected {accumulated_raw_mass:.2f} tons out of required {target_quota_tons} tons.")

## Run the whole sequence

`main()` loads the data and runs Steps 1 to 6 in order. Change the arguments to re-run with different inputs.

In [9]:
def main(file_path=FILE_PATH,
         high_value_threshold=HIGH_VALUE_THRESHOLD,
         target_quota_tons=TARGET_QUOTA_TONS):
    shipments = load_shipments(file_path)

    # STEP 1
    port_fob_totals = calculate_port_fob_totals(shipments)
    print_port_fob_totals(port_fob_totals)

    # STEP 2
    high_value_shipments = filter_high_value_shipments(shipments, high_value_threshold)
    print_high_value_summary(high_value_shipments, high_value_threshold)

    # STEP 3
    classified_shipments = classify_product_types(high_value_shipments)
    print_product_classification(classified_shipments)

    # STEP 4
    assigned_shipments = assign_audit_protocols(classified_shipments)
    print_compliance_tiers(assigned_shipments)

    # STEP 5
    audit_sample_batch, accumulated_raw_mass = accumulate_audit_sample(assigned_shipments, target_quota_tons)
    print_audit_sample_summary(audit_sample_batch, accumulated_raw_mass)

    # STEP 6
    print_quota_verification(audit_sample_batch, accumulated_raw_mass, target_quota_tons)


main()

=== STEP 1: PORT FOB TOTALS ===
Port: BUENAVENTURA                   | Total FOB: $190,769.12
Port: SANTOS                         | Total FOB: $374,245.29
Port: RIO DE JANEIRO                 | Total FOB: $236,566.06
Port: BELAWAN                        | Total FOB: $75,301.78
Port: ESPECIAL DE CARTAGENA          | Total FOB: $9,537.03
Port: CAI MEP PORT - TCIT (VUNG TAU) | Total FOB: $99,338.40
Port: PANJANG                        | Total FOB: $57,599.51
Port: PUERTO DEL CALLAO              | Total FOB: $1,988.22

=== STEP 2: HIGH-VALUE FILTER ===
Isolated 6 shipments exceeding $50,000 FOB.

=== STEP 3: PRODUCT TYPE CLASSIFICATION ===
HS 090111 (Unroasted Green Coffee Beans) | FOB: $75,301.78 -> Destination: SOUTH KOREA
HS 210111 (Instant Coffee Extracts & Derivatives) | FOB: $109,947.90 -> Destination: UNITED STATES
HS 090111 (Unroasted Green Coffee Beans) | FOB: $99,338.40 -> Destination: GERMANY
HS 090111 (Unroasted Green Coffee Beans) | FOB: $74,823.41 -> Destination: ITALY
HS 09